
# Spell Check dengan Python

## Pendahuluan

Spell check (pemeriksaan ejaan) adalah proses dalam pengolahan bahasa alami (Natural Language Processing - NLP) untuk mendeteksi dan mengoreksi kesalahan penulisan (typo) dalam teks. Dalam modul ini, kita akan menggunakan pustaka `textdistance` untuk membandingkan kemiripan antara kata yang salah dengan kata yang benar berdasarkan kamus yang telah ditentukan.

## Proses Spell Check

Proses spell check secara umum terdiri dari langkah-langkah berikut:

1. **Input Data**: Membaca teks mentah atau kumpulan kalimat dari file eksternal.
2. **Tokenisasi**: Memecah kalimat menjadi kata-kata.
3. **Perbandingan dengan Kamus**: Setiap kata yang tidak cocok dengan kamus akan dibandingkan jaraknya dengan seluruh kata dalam kamus.
4. **Koreksi**: Kata diganti dengan kata dalam kamus yang memiliki *similarity score* tertinggi.
5. **Rekonstruksi Teks**: Kata-kata hasil koreksi disatukan kembali menjadi kalimat.

### Contoh Pemanfaatan Spell Check
1. Cek plagiarisme
2. Cek typo/mispelled words
3. Pencarian kata kunci di web search/e-commerce
4. Pencarian kemiripan nama
5. dll.

### Metode
Metode pengukuran kemiripan kata yang umum digunakan meliputi:
- **Levenshtein Distance**: Menghitung jumlah perubahan (insert, delete, replace) minimum yang dibutuhkan untuk mengubah satu string menjadi string lain.
- **Jaro-Winkler** dan **Hamming** juga merupakan metode populer lainnya.

Kita akan menggunakan fungsi dari `textdistance` untuk menghitung jarak Levenshtein sebagai dasar koreksi ejaan.




## A. Pengenalan Library Levenshtein Distance

In [1]:
pip install python-Levenshtein


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [python-Levenshtein]


In [2]:
!pip install textdistance

In [3]:
import numpy as np
import pandas as pd

teks2 = "sangat"
teks1 = "sangad"

def levenshtein_matrix(s1, s2):
    rows = len(s1) + 1
    cols = len(s2) + 1

    # Inisialisasi matriks
    matrix = [[0 for _ in range(cols)] for _ in range(rows)]

    # Isi baris dan kolom pertama
    for i in range(rows):
        matrix[i][0] = i
    for j in range(cols):
        matrix[0][j] = j

    # Hitung isi matriks
    for i in range(1, rows):
        for j in range(1, cols):
            if s1[i-1] == s2[j-1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i-1][j] + 1,      # delete
                matrix[i][j-1] + 1,      # insert
                matrix[i-1][j-1] + cost  # substitute
            )

    return matrix


def print_matrix(matrix, s1, s2):
    print("  ", "  ".join([" "] + list(s2)))
    for i, row in enumerate(matrix):
        if i == 0:
            label = " "
        else:
            label = s1[i-1]
        print(label, row)

matrix = levenshtein_matrix(teks1, teks2)
print_matrix(matrix, teks1, teks2)

      s  a  n  g  a  t
  [0, 1, 2, 3, 4, 5, 6]
s [1, 0, 1, 2, 3, 4, 5]
a [2, 1, 0, 1, 2, 3, 4]
n [3, 2, 1, 0, 1, 2, 3]
g [4, 3, 2, 1, 0, 1, 2]
a [5, 4, 3, 2, 1, 0, 1]
d [6, 5, 4, 3, 2, 1, 1]


In [4]:
import Levenshtein

# Menghitung jarak Levenshtein
distance = Levenshtein.distance("sangad", "sangat")
print("Distance:", distance)

Distance: 1


In [5]:
import Levenshtein
Levenshtein.distance("kucing", "kacing")

1

In [6]:
import textdistance
dist_lev = textdistance.levenshtein.distance("kucing", "kacing")          # hasil: 1
sim_lev = textdistance.levenshtein.normalized_similarity("kucing", "kacing")  # hasil: 0.83

# Misal mau pakai metode pengukuran lainnya (Jaro-Winkler)
dist_jaro = textdistance.jaro.distance("kucing", "kacing")
sim_jaro  = textdistance.jaro.normalized_similarity("kucing", "kacing")

print("Distance pakai Levenshtein: ", dist_lev)
print("Similarity Levenshtein: ", sim_lev)
print()
print("Distance pakai Jaro: ", dist_jaro)
print("Similarity Jaro: ", sim_jaro)


Distance pakai Levenshtein:  1
Similarity Levenshtein:  0.8333333333333334

Distance pakai Jaro:  0.11111111111111105
Similarity Jaro:  0.888888888888889


In [7]:
corpus = ["kucing", "anjing", "kuda", "kelinci", "sapi", "kambing", "gajah"]
misspelled = ["kuncing", "anjin", "kuuda", "kelici", "sappi", "kmbing", "gajjah"]

In [8]:
#corpus: daftar kata benar.
#misspelled: daftar kata salah eja yang akan dikoreksi.

In [9]:
def correct_levenshtein(word, vocab):
    return min(vocab, key=lambda x: textdistance.levenshtein.distance(word, x))

In [10]:
results = []
for w in misspelled:
    results.append({
        "misspelled": w,
        "levenshtein": correct_levenshtein(w, corpus),

    })
results

[{'misspelled': 'kuncing', 'levenshtein': 'kucing'},
 {'misspelled': 'anjin', 'levenshtein': 'anjing'},
 {'misspelled': 'kuuda', 'levenshtein': 'kuda'},
 {'misspelled': 'kelici', 'levenshtein': 'kelinci'},
 {'misspelled': 'sappi', 'levenshtein': 'sapi'},
 {'misspelled': 'kmbing', 'levenshtein': 'kambing'},
 {'misspelled': 'gajjah', 'levenshtein': 'gajah'}]

## B. Praktik Spell Check Menggunakan Data Tweet Efisiensi

In [11]:
import pandas as pd
import textdistance

### B1. Siapkan Dataset

In [12]:
!wget https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/tweet_efisiensi.csv

--2026-10-27 10:30:12--  https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/tweet_efisiensi.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 865314 (845K) [text/plain]
Saving to: ‘tweet_efisiensi.csv’

tweet_efisiensi.csv 100%[===================>] 845.03K  5.08MB/s    in 0.2s    

2026-10-27 10:30:13 (5.08 MB/s) - ‘tweet_efisiensi.csv’ saved [865314/865314]



In [13]:
# 2. Baca file CSV (pastikan file sudah di-upload sebelumnya)
df = pd.read_csv('tweet_efisiensi.csv')  # ganti nama file jika perlu

### B2. Definisikan Kamus Kata

In [14]:
# Definisikan kamus kata (bisa ditambah sesuai kebutuhan)
kamus = ['efisiensi', 'logika', 'dana', 'penghematan', 'anggaran',
         'kebijakan', 'publik', 'transparansi', 'negara', 'rakyat', 'subsidi',
         'duit', 'uang', 'biaya', 'salah', 'silap', 'banyak', 'omong']

### B3. Buat Fungsi untuk Koreksi Kata

In [15]:
# 4. Fungsi untuk koreksi kata. Kata akan dicocokkan dengan data pada kamus yang didefinisikan di atas
def koreksi_kata(word, vocab):
    return min(vocab, key=lambda x: textdistance.levenshtein.normalized_distance(word.lower(), x))

# 5. Fungsi untuk koreksi seluruh teks
def correct_text(text, vocab):
    words = text.split()
    corrected = [
        koreksi_kata(word, vocab) if word.lower() not in vocab else word
        for word in words
    ]
    return ' '.join(corrected)

### B4. Proses Koreksi Kata dan Hasilnya

In [16]:

# 6. Ambil 10 tweet contoh dari kolom 'full_text'
tweets = df['full_text'].dropna().sample(10, random_state=42).tolist()

# 7. Tampilkan hasil koreksi
for i, tweet in enumerate(tweets):
    print(f"Tweet {i+1} Asli     : {tweet}")
    print(f"Tweet {i+1} Koreksi  : {correct_text(tweet, kamus)}")
    print("-" * 50)

Tweet 1 Asli     : efisiensi alibi kontrak ga diperpanjang adalah MONYEEEEET 
Tweet 1 Koreksi  : efisiensi subsidi banyak logika dana salah negara
--------------------------------------------------
Tweet 2 Asli     : @aoreazudan Orang yg manage twitter nya kan bukan elon ajaa. Mungkin mereka kena efisiensi
Tweet 2 Koreksi  : anggaran uang uang dana duit banyak dana kebijakan omong anggaran publik logika dana efisiensi
--------------------------------------------------
Tweet 3 Asli     : @tarnesolea @unsoedmfs kasian gbs pake lift krn efisiensi https://t.co/NCFGjNbrh5
Tweet 3 Koreksi  : dana subsidi kebijakan subsidi rakyat logika dana efisiensi anggaran
--------------------------------------------------
Tweet 4 Asli     : Puasa bukan hanya sekadar ritual spiritual tetapi juga memiliki dampak signifikan pada metabolisme tubuh kita. Salah satu manfaat paling menonjol dari puasa adalah kemampuannya untuk meningkatkan efisiensi pembakaran lemak. https://t.co/esxh4DgHo6
Tweet 4 Koreksi  : d

Karena keterbatasan kamus kata yang didefinisikan manual di atas, maka hasil Tweet Koreksi kebanyakan tidak sesuai dengan yang kita harapkan (sesuai dengan pengetahuan kita). Program hanya akan mendeteksi kumpulan kamus kata yang telah didefinisikan sebelumnya.

## C. Hasil Spell Check Jika Melalui Tahap Preprocessing

In [17]:
df_clean = df.copy()

In [18]:
df_clean = df_clean.dropna().sample(10, random_state=42)
df_clean

# Karena menggunakan random_state, maka 10 data yang ditampilkan di bawah kemungkinan akan berbeda dengan 10 data sebelumnya (dipilih sample secara acak)

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
2144,1898940913566490742,Mon Mar 10 03:36:09 +0000 2025,0,... Pusat Data dan Informasi Pertanahan (PUSDA...,1898940916922302490,https://pbs.twimg.com/media/Glpkc6FbgAAxdHc.jpg,kantahkabpasbar,in,"Sumatera Barat, Indonesia",0,0,0,https://x.com/kantahkabpasbar/status/189894091...,1086875203995459584,kantahkabpasbar
942,1899676638306062839,Wed Mar 12 04:51:33 +0000 2025,0,@tarnesolea @unsoedmfs kasian gbs pake lift kr...,1899684664081854853,https://pbs.twimg.com/media/Gl0JIDpXwAA4nk5.jpg,tarnesolea,in,"Gravity Falls, Oregon",0,0,0,https://x.com/1979rhymes/status/18996846640818...,1812701272707145728,1979rhymes
2063,1899017889606402362,Mon Mar 10 08:42:32 +0000 2025,0,Tingkatkan efisiensi klinik Anda dan berikan p...,1899018020141576366,https://pbs.twimg.com/media/GlqqxnVbwAAceM2.jpg,KsatriaMedical,in,"Australia, Indonesia",0,0,0,https://x.com/KsatriaMedical/status/1899018020...,953144252661317633,KsatriaMedical
1094,1899320608296665175,Tue Mar 11 21:38:19 +0000 2025,1,@txtdrimedia Ini poli-tikus apa ngak hapal dan...,1899575639579193763,https://pbs.twimg.com/media/Glyl-lpXQAAac3Y.jpg,txtdrimedia,in,Jakarta Capital Region,0,0,0,https://x.com/bakanosan1/status/18995756395791...,1604650474724265985,bakanosan1
439,1899960013004955680,Thu Mar 13 06:06:05 +0000 2025,0,@zainalamochtar Mulai dari sini juga dong efis...,1900065812695634333,https://pbs.twimg.com/media/Gl5jxlGacAAeY0p.jpg,zainalamochtar,in,ada dimana mana,0,0,0,https://x.com/Graminale/status/190006581269563...,1071545084,Graminale
446,1899780420298031324,Thu Mar 13 05:57:51 +0000 2025,0,@geloraco Demi Efisiensi.... kedengarannya san...,1900063739614736483,https://pbs.twimg.com/media/Gl5h5ueaoAAvwuI.png,geloraco,in,"Kediri, Jawa Timur",0,0,0,https://x.com/red_troa/status/1900063739614736483,701327459313004544,red_troa
2155,1898936709368398030,Mon Mar 10 03:19:26 +0000 2025,0,@infoBMKG hi Admin citra radar cuaca di https:...,1898936709368398030,https://pbs.twimg.com/media/GlpgiUTakAA-Ukn.jpg,infoBMKG,in,6.5 inches Super AMOLED screen,0,0,0,https://x.com/tomikalaza/status/18989367093683...,49465717,tomikalaza
2083,1898981195914019032,Mon Mar 10 06:56:37 +0000 2025,0,@PartaiSocmed Pasti karena INPRES EFISIENSI 25...,1898991362738999530,https://pbs.twimg.com/media/GlqSlMkXkAAvHfB.jpg,PartaiSocmed,in,East Renfrewshire,0,0,0,https://x.com/JPGunarto/status/189899136273899...,1741986317079056384,JPGunarto
811,1899753457755767195,Wed Mar 12 09:25:02 +0000 2025,3,9/ Mengurangi Penundaan Lalu Lintas Udara Memu...,1899753489888403566,https://pbs.twimg.com/media/Gl1HRRybkAAmyXO.jpg,cryptondo,in,Oasis,0,1,0,https://x.com/cryptondo/status/189975348988840...,1471393708478058505,cryptondo
13,1900319243738206348,Thu Mar 13 23:11:27 +0000 2025,0,@milskymelon (masih si pink) hmzz pdhal klow 1...,1900323852468838551,https://pbs.twimg.com/tweet_video_thumb/Gl9Od0...,milskymelon,in,"Melbourne, Victoria",0,1,0,https://x.com/remleemark/status/19003238524688...,926035792329240576,remleemark


In [19]:
!pip install Sastrawi

In [20]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
stopwords_ind = stopwords.words('indonesian')

import re  # Import the 're' module for regular expressions
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Import Sastrawi for stemming

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [21]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Membuat function untuk cleaning data
def clean_text(text):
  text = re.sub(r'http\S+', '', text)         # Remove URLs
  text = re.sub(r'@\S+', '', text)            # Remove mentions (@username)
  text = re.sub(r'#\S+', '', text)            # Remove hashtags (#hashtag)
  text = re.sub(r'[^a-zA-Z\s]', '', text)     # Remove special characters and numbers
  text = re.sub(r'\s+', ' ', text).strip()    # Remove extra whitespaces
  return text

def casefolding(text):
  text = text.lower()                               # Mengubah teks menjadi lower case
  return text

def remove_stop_words(text):
  clean_words = []
  text = text.split()
  for word in text:
      if word not in stopwords_ind:
          clean_words.append(word)
  return " ".join(clean_words)

# Buat fungsi untuk langkah stemming bahasa Indonesia
def text_stemming(text):
  text = stemmer.stem(text)
  return text

def text_preprocessing_process(text):
  text = clean_text(text)
  text = casefolding(text)
  text = remove_stop_words(text)
  text = text_stemming(text)
  return text



In [22]:
df_clean['cleaned_text'] = df_clean['full_text'].apply(text_preprocessing_process)
df_clean[['full_text', 'cleaned_text']]

,full_text,cleaned_text
2144,... Pusat Data dan Informasi Pertanahan (PUSDA...,pusat data informasi tanah pusdatin ada sosial...
942,@tarnesolea @unsoedmfs kasian gbs pake lift kr...,kasi gbs pake lift krn efisiensi
2063,Tingkatkan efisiensi klinik Anda dan berikan p...,tingkat efisiensi klinik layan baik pasien dm ...
1094,@txtdrimedia Ini poli-tikus apa ngak hapal dan...,politikus ngak hapal dana didik sehat udah efi...
439,@zainalamochtar Mulai dari sini juga dong efis...,efisiensi nya coretax masalah kek gak fungsi d...
446,@geloraco Demi Efisiensi.... kedengarannya san...,efisiensi dengar merdu
2155,@infoBMKG hi Admin citra radar cuaca di https:...,hi admin citra radar cuaca minggu akurat ya wi...
2083,@PartaiSocmed Pasti karena INPRES EFISIENSI 25...,inpres efisiensi januari kemarin nih
811,9/ Mengurangi Penundaan Lalu Lintas Udara Memu...,kurang tunda lintas udara koordinasi pesawat k...
13,@milskymelon (masih si pink) hmzz pdhal klow 1...,si pink hmzz pdhal klow yeti bungkus pc na kt ...


In [23]:
# 6. Ambil 5 tweet contoh dari kolom 'full_text'
tweets_clean = df_clean['cleaned_text'].dropna().sample(5, random_state=42).tolist()

# 7. Tampilkan hasil koreksi
for i, tweets in enumerate(tweets_clean):
    print(f"Tweet {i+1} Asli     : {tweets}")
    print(f"Tweet {i+1} Koreksi  : {correct_text(tweets, kamus)}")
    print("-" * 50)

Tweet 1 Asli     : kurang tunda lintas udara koordinasi pesawat kendali kurang tunda tingkat efisiensi lintas udara teknologi adsb main peran tingkat selamat
Tweet 1 Koreksi  : uang dana dana dana efisiensi negara kebijakan uang dana penghematan efisiensi dana dana transparansi salah penghematan anggaran penghematan negara
--------------------------------------------------
Tweet 2 Asli     : kasi gbs pake lift krn efisiensi
Tweet 2 Koreksi  : subsidi subsidi rakyat logika dana efisiensi
--------------------------------------------------
Tweet 3 Asli     : efisiensi dengar merdu
Tweet 3 Koreksi  : efisiensi dana negara
--------------------------------------------------
Tweet 4 Asli     : pusat data informasi tanah pusdatin ada sosialisasi kait guna librecad tingkat efisiensi kelola data tanah
Tweet 4 Koreksi  : duit dana transparansi dana publik dana efisiensi duit dana logika penghematan efisiensi kebijakan dana dana
--------------------------------------------------
Tweet 5 Asli     :

Karena keterbatasan kamus kata yang didefinisikan manual di atas, maka hasil Tweet Koreksi kebanyakan tidak sesuai dengan yang kita harapkan (sesuai dengan pengetahuan kita). Program hanya akan mengoreksi kesamaan kata berdasarkan kumpulan kamus kata yang telah didefinisikan sebelumnya.

```
kamus = ['efisiensi', 'logika', 'dana', 'penghematan', 'anggaran',
         'kebijakan', 'publik', 'transparansi', 'negara', 'rakyat', 'subsidi',
         'duit', 'uang', 'biaya', 'salah', 'silap', 'banyak', 'omong']
```



## D. Tambahan: Cek Skor Similarity Setiap Kata

Untuk melihat perbandingan skor kesamaan (similarity score) antara 2 kalimat, kita bisa menggunakan format matriks

In [24]:
import textdistance

def levenshtein_similarity_matrix(text1, text2):
    words1 = text1.split()
    words2 = text2.split()

    matrix = []
    for w1 in words1:
        row = []
        for w2 in words2:
            sim = textdistance.levenshtein.normalized_similarity(w1, w2)
            row.append(round(sim, 2))
        matrix.append(row)
    return words1, words2, matrix

In [25]:
# Masukkan contoh kalimat, misal ambil dari hasil Teks Asli 2 dan Teks Koreksi 2
teks_asli = "kasi gbs pake lift krn efisiensi"
teks_koreksi = "subsidi subsidi rakyat logika dana efisiensi"

words1, words2, sim_matrix = levenshtein_similarity_matrix(teks_asli, teks_koreksi)

# Display matrix
import pandas as pd
matrix_kesamaan = pd.DataFrame(sim_matrix, index=words1, columns=words2)
print(matrix_kesamaan)


           subsidi  subsidi  rakyat  logika  dana  efisiensi
kasi          0.29     0.29    0.17    0.17  0.25       0.22
gbs           0.29     0.29    0.00    0.17  0.00       0.11
pake          0.00     0.00    0.33    0.17  0.25       0.11
lift          0.14     0.14    0.17    0.33  0.00       0.11
krn           0.00     0.00    0.17    0.00  0.25       0.11
efisiensi     0.33     0.33    0.00    0.11  0.11       1.00


##**POST TEST**

1. Berikan pendapatmu terkait hasil akhir di file python yaitu perbandingan antara tweet asli dan tweet koreksi. Nyatakan kelemahan dari hasil tersebut.
2. Lakukan eksperimen similarity matrix dengan metode selain Levenshtein!.

Jawaban 1: Analisis Hasil Akhir & Kelemahan Metode

Pendapat & Analisis:
Hasil perbandingan antara tweet asli dan tweet hasil koreksi menunjukkan performa yang sangat kurang memuaskan dan tidak natural. Banyak kata yang maknanya berubah total menjadi kata lain yang tidak relevan secara konteks semantik (misalnya, kata "kasi" dikoreksi menjadi "subsidi",).

kelemahan utama:
1. keterbatasan kamus: Program ini hanya memiliki daftar kata yang sangat sedikit (hanya 18 kata). Akibatnya, setiap kata di luar kamus dipaksa untuk dipetakan ke salah satu dari 18 kata tersebut, meskipun jarak Levenshtein-nya sangat jauh dan maknanya tidak menyambung.
2. ketidakadaan konteks semantik. Jarak Levenshtein murni mengukur kemiripan karakter per karakter tanpa memperdulikan konteks kalimat
3. kata singkatan dan slang. Tweet bahasa Indonesia tu banyak singkatan (gbs, krn, nih, dll) yang tidak terdaftar di KBBI standard maupun kamus yang dibuat disini, sehingga menghasilkan proses pemetaan paksa yang salah.

In [27]:
import textdistance
import pandas as pd

def jaro_winkler_similarity_matrix(text1, text2):
    words1 = text1.split()
    words2 = text2.split()

    matrix = []
    for w1 in words1:
        row = []
        for w2 in words2:
            sim = textdistance.jaro_winkler.normalized_similarity(w1, w2)
            row.append(round(sim, 2))
        matrix.append(row)
    return words1, words2, matrix

teks_asli = "kasi gbs pake lift krn efisiensi"
teks_koreksi = "subsidi subsidi rakyat logika dana efisiensi"

words1, words2, sim_matrix_jaro = jaro_winkler_similarity_matrix(teks_asli, teks_koreksi)

matrix_kesamaan_jaro = pd.DataFrame(sim_matrix_jaro, index=words1, columns=words2)
print("=== Matriks Kemiripan menggunakan Jaro-Winkler ===")
display(matrix_kesamaan_jaro)

=== Matriks Kemiripan menggunakan Jaro-Winkler ===


,subsidi,subsidi,rakyat,logika,dana,efisiensi
kasi,0.60,0.60,0.44,0.47,0.50,0.41
gbs,0.48,0.48,0.00,0.50,0.00,0.48
pake,0.00,0.00,0.61,0.47,0.50,0.45
lift,0.00,0.00,0.47,0.61,0.00,0.41
krn,0.00,0.00,0.50,0.00,0.53,0.00
efisiensi,0.48,0.48,0.00,0.43,0.00,1.00
